# Agentic AI & RAG Engineering — Course Notebook (Weeks 1–3)

**Author:** Narayanan Palani  
**Scope:** Weeks 1 to 3 Complete Implementation Reference  
**Stack:** Python, OpenAI (`gpt-4o-mini`), Pydantic v2, AsyncIO, SQLite, FastAPI, Streamlit, Pytest

--- 
## 1. Environment Setup & Dependency Configuration
* **Slide Source:** *Week 1 - Slide 17 ("Setup — Two Minutes"), Week 2 - Slide 26 ("Secrets — Extending the W1 Discipline")*
* **Action:** Install required packages and securely load API credentials using `python-dotenv`.

In [9]:
%pip install ollama
import ollama

response = ollama.chat(
    model="gpt-oss:20b",
    messages=[
        {
            "role": "user",
            "content": "What is the core benefit of RAG? Answer with exactly one word."
        }
    ]
)

print(response.message.content)

Note: you may need to restart the kernel to use updated packages.


ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download

In [5]:
# Slide Source:
#   Week 1 - Slide 17 ("Setup — Two Minutes")
#   Week 2 - Slide 26 ("Secrets — Extending the W1 Discipline")
#
# Purpose:
#   1. Import the libraries needed to work with environment variables.
#   2. Load secrets stored in a local .env file.
#   3. Retrieve the OpenAI API key without hard-coding it in the notebook.
#   4. Validate that the API key is available before making API calls.
#
# Security principle:
#   Never hard-code an API key directly into Python source code.
#   Never print the actual API key.
#   Keep .env out of source control by adding it to .gitignore.


# os provides access to environment variables through os.getenv().
import os


# python-dotenv allows Python to read variables from a local .env file
# and make them available through the operating system environment.
from dotenv import load_dotenv


# Read the .env file from the current working directory.
#
# For example, your .env file should contain:
#
# OPENAI_API_KEY=sk-your-api-key-here
#
# load_dotenv() does NOT print or expose the secret.
load_dotenv()


# Retrieve the OpenAI API key from the environment.
#
# os.getenv() returns:
#   - The value of OPENAI_API_KEY if it exists.
#   - None if the variable has not been defined.
#
# We store the key in a Python variable so it can later be passed
# to the OpenAI client.
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")


# Validate the configuration before attempting an API call.
#
# This is useful because it gives us a clear configuration error
# instead of allowing a later API call to fail with a less obvious error.
if not OPENAI_API_KEY:
    raise ValueError(
        "OPENAI_API_KEY not found. "
        "Please add it to your .env file."
    )


# Confirm that the environment has been configured successfully.
#
# IMPORTANT:
# We deliberately do NOT print the API key itself.
# API keys are credentials and should be treated like passwords.
print("Environment successfully initialized. API key detected.")

Environment successfully initialized. API key detected.


--- 
## 2. Week 1: Foundations, Decision Frameworks & Hello LLM
* **Slide Source:** *Week 1 - Slide 06 ("A Working Definition"), Slide 11-16 ("Four Patterns"), Slide 17-18 ("Hello LLM & Lab Step 2")*
* **Action:** Initialize standard OpenAI client, run a baseline API prompt using `gpt-4o-mini`, and log token usage metrics.

In [7]:
# Import the OpenAI client.
from openai import OpenAI


# Create the client.
# The SDK reads OPENAI_API_KEY from your environment.
client = OpenAI()


# Make a lightweight request to the Models API.
#
# This verifies that:
#   1. The API endpoint is reachable.
#   2. Your API key is accepted.
#   3. Your key has access to the API.
#
# This does NOT ask an LLM to generate a response.
models = client.models.list()


# If we reached this point, authentication succeeded.
print("OpenAI API connection successful.")


# Show a few models available to your API key.
for model in models.data[:5]:
    print(model.id)

OpenAI API connection successful.
text-embedding-ada-002
whisper-1
gpt-3.5-turbo
tts-1
gpt-3.5-turbo-16k


In [6]:
# Install the OpenAI Python SDK once in this notebook environment.
# The "-q" flag keeps installation output quiet.
# Uncomment and run this only if the package is not already installed.
# %pip install -q openai


# Import the OpenAI client class from the OpenAI Python SDK.
from openai import OpenAI


# Create one OpenAI client for the notebook.
# The client automatically reads OPENAI_API_KEY from the environment.
#
# Your .env file should contain:
# OPENAI_API_KEY=your_api_key_here
#
# If you are using a .env file, make sure you have loaded it separately
# with python-dotenv:
#
# from dotenv import load_dotenv
# load_dotenv()
#
client = OpenAI()


def run_hello_llm(prompt_text: str) -> str:
    """
    Send a prompt to the OpenAI API and return the model's response.

    prompt_text:
        The question/instruction that will be sent to the model.

    Returns:
        The text generated by the model.
    """

    # Send a request to the Chat Completions API.
    response = client.chat.completions.create(

        # Select the model used for this experiment.
        # gpt-4o-mini is a relatively small, cost-efficient model.
        model="gpt-4o-mini",

        # Provide the conversation sent to the model.
        # We use only a user message here to minimize unnecessary
        # prompt tokens.
        messages=[
            {
                "role": "user",
                "content": prompt_text
            }
        ],

        # Limit the maximum number of tokens the model can generate.
        # Since our question asks for one word, 5 tokens is more than enough.
        # This helps prevent unnecessarily long completions.
        max_tokens=5,

        # Lower temperature makes the response more deterministic.
        # A value of 0.2 is appropriate for a simple factual response.
        temperature=0.2
    )

    # Extract the API usage information from the response.
    # This lets us see how many tokens were consumed.
    usage = response.usage

    # Display token usage for this API call.
    #
    # prompt_tokens:
    #   Tokens sent to the model.
    #
    # completion_tokens:
    #   Tokens generated by the model.
    #
    # total_tokens:
    #   Prompt + completion tokens.
    print(
        f"Prompt tokens: {usage.prompt_tokens} | "
        f"Completion tokens: {usage.completion_tokens} | "
        f"Total tokens: {usage.total_tokens}"
    )

    # Extract only the text generated by the model.
    content = response.choices[0].message.content

    # Return the generated text to the caller.
    return content


# Define the actual prompt we want to send to the model.
#
# Keeping the prompt short reduces input-token usage.
# The instruction "exactly one word" also helps keep the response concise.
prompt = "What is the core benefit of RAG? Answer with exactly one word."


# Call our function and store the model's response.
result = run_hello_llm(prompt)


# Display the final response returned by the model.
print("\nLLM Response:")
print(result)

RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

--- 
## 3. Week 2: Typed Contracts (Pydantic) & Async Concurrency Pipeline
* **Slide Source:** *Week 2 - Slide 05-07 ("Pydantic Models"), Slide 09-11 ("Async Basics & httpx"), Slide 15-18 ("Concurrency Patterns: Gather, Batching, Retry"), Slide 23 ("Structured Logging"), Slide 27 ("SQLite Store")*
* **Action:** Build Pydantic schemas, an async client with exponential retry backoff, parallel batch processing via `asyncio.gather`, JSON structured logging, and SQLite persistence.

In [ ]:
# Slide Source: Week 2 - Slide 05-07 ("Pydantic Models"), Slide 09-11 ("Async Basics & httpx"), 
# Slide 15-18 ("Concurrency Patterns: Gather, Batching, Retry"), Slide 23 ("Structured Logging"), Slide 27 ("SQLite Store")
# Action: Build Pydantic schemas, an async client with exponential retry backoff, parallel batch processing via asyncio.gather, JSON structured logging, and SQLite persistence.

import asyncio
import json
import logging
import sqlite3
import time
from typing import List, Optional
from pydantic import BaseModel, Field
from openai import AsyncOpenAI

# Structured Logging Setup
logging.basicConfig(level=logging.INFO, format='%(message)s')

def log_json(event: str, **kwargs):
    log_entry = {"event": event, "timestamp": time.time(), **kwargs}
    logging.info(json.dumps(log_entry))

# Pydantic Schemas
class QuestionRequest(BaseModel):
    id: int
    query: str = Field(..., min_length=3, description="The user query")

class AnswerResponse(BaseModel):
    id: int
    query: str
    answer: str
    status: str = "success"

# SQLite Persistence Initialization
def init_db():
    conn = sqlite3.connect("results.db")
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS run_results (
            id INTEGER PRIMARY KEY,
            query TEXT NOT NULL,
            answer TEXT NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)
    conn.commit()
    conn.close()

def save_to_db(record: AnswerResponse):
    conn = sqlite3.connect("results.db")
    cursor = conn.cursor()
    cursor.execute(
        "INSERT INTO run_results (id, query, answer) VALUES (?, ?, ?)",
        (record.id, record.query, record.answer)
    )
    conn.commit()
    conn.close()

# Retry Decorator / Wrapper with Async Processing
async def with_retry(coro_func, *args, max_retries: int = 3, **kwargs):
    for attempt in range(1, max_retries + 1):
        try:
            return await coro_func(*args, **kwargs)
        except Exception as exc:
            log_json("async_retry_attempt", attempt=attempt, max_retries=max_retries, error=str(exc))
            if attempt == max_retries:
                raise exc
            await asyncio.sleep(2 ** attempt)

async def process_single_query(async_client: AsyncOpenAI, req: QuestionRequest) -> AnswerResponse:
    async def _call():
        response = await async_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": req.query}],
            temperature=0.3
        )
        return response.choices[0].message.content.strip()

    answer_text = await with_retry(_call)
    res = AnswerResponse(id=req.id, query=req.query, answer=answer_text)
    save_to_db(res)
    log_json("query_processed", id=req.id, query=req.query)
    return res

async def batch_process_pipeline(queries: List[QuestionRequest]):
    init_db()
    async_client = AsyncOpenAI(api_key=OPENAI_API_KEY)
    
    tasks = [process_single_query(async_client, q) for q in queries]
    results = await asyncio.gather(*tasks, return_exceptions=True)
    return results

# Driver Code
if __name__ == "__main__":
    sample_queries = [
        QuestionRequest(id=1, query="What is Pydantic in Python?"),
        QuestionRequest(id=2, query="How does asyncio.gather speed up API calls?"),
        QuestionRequest(id=3, query="Why use SQLite for local execution persistence?")
    ]
    
    print("Starting Async Batch Pipeline...")
    batch_results = await batch_process_pipeline(sample_queries) if 'get_ipython' in globals() else asyncio.run(batch_process_pipeline(sample_queries))
    for res in batch_results:
        print(f"\nID: {res.id}\nQuery: {res.query}\nAnswer: {res.answer[:100]}...")

--- 
## 4. Week 3: FastAPI Web Service & Streaming Endpoint
* **Slide Source:** *Week 3 - Slide 06 ("FastAPI 12-line App"), Slide 08 ("/health Endpoint"), Slide 12-14 ("Streaming via StreamingResponse")*
* **Action:** Write `main.py` containing FastAPI backend supporting a health probe, structured POST endpoint, and streaming token response via SSE / raw stream.

In [ ]:
%%writefile main.py
# Slide Source: Week 3 - Slide 06 ("FastAPI 12-line App"), Slide 08 ("/health Endpoint"), Slide 12-14 ("Streaming via StreamingResponse")
# Action: Build FastAPI backend supporting a health probe, structured POST endpoint, and streaming token response via SSE / raw stream.

from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from openai import AsyncOpenAI
import asyncio
import os

app = FastAPI(title="Agentic RAG Engine API", version="1.0.0")
async_client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

class AskRequest(BaseModel):
    query: str

@app.get("/health")
async def health_check():
    return {"status": "ok", "service": "agentic-rag-engine"}

@app.post("/ask")
async def ask_endpoint(request: AskRequest):
    if not request.query.strip():
        raise HTTPException(status_code=400, detail="Query string cannot be empty.")
    
    response = await async_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": request.query}]
    )
    return {"query": request.query, "answer": response.choices[0].message.content}

async def stream_generator(query: str):
    response = await async_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": query}],
        stream=True
    )
    async for chunk in response:
        content = chunk.choices[0].delta.content
        if content:
            yield content

@app.post("/stream")
async def stream_endpoint(request: AskRequest):
    return StreamingResponse(stream_generator(request.query), media_type="text/plain")

# Run locally with command: uvicorn main:app --reload --port 8000

--- 
## 5. Streamlit Frontend UI
* **Slide Source:** *Week 3 - Slide 16 ("Minimal Streamlit UI")*
* **Action:** Write `app.py` constructing UI consuming the FastAPI streaming response in real-time.

In [ ]:
%%writefile app.py
# Slide Source: Week 3 - Slide 16 ("Minimal Streamlit UI")
# Action: Construct UI consuming the FastAPI streaming response in real-time.

import streamlit as st
import requests

st.set_page_config(page_title="Agentic RAG Control Center", layout="wide")
st.title("Agentic AI & RAG Interface")

query_input = st.text_input("Enter your request or prompt:", placeholder="Ask something...")

if st.button("Submit Query"):
    if not query_input.strip():
        st.warning("Please enter a valid query.")
    else:
        st.subheader("Streaming Response:")
        response_box = st.empty()
        full_response = ""
        
        try:
            url = "http://localhost:8000/stream"
            with requests.post(url, json={"query": query_input}, stream=True) as response:
                if response.status_code == 200:
                    for chunk in response.iter_content(chunk_size=1024, decode_unicode=True):
                        if chunk:
                            full_response += chunk
                            response_box.markdown(full_response + "▌")
                    response_box.markdown(full_response)
                else:
                    st.error(f"Error {response.status_code}: Unable to reach API.")
        except Exception as e:
            st.error(f"Connection error: {str(e)}")

# Run with command: streamlit run app.py

--- 
## 6. Week 3 (Day 2): Testing, Mocks & API Contract Specification
* **Slide Source:** *Week 3 - Slide 28-30 ("Testing & Mocks"), Slide 33-36 ("API Contracts & ADR 0002")*
* **Action:** Unit test pipeline components with `pytest` & `AsyncMock`, and document Architectural Decision Record (ADR 0002).

In [ ]:
# Slide Source: Week 3 - Slide 28-30 ("Testing & Mocks")
# Action: Test execution components without making external API network requests using AsyncMock.

import pytest
from unittest.mock import AsyncMock
from pydantic import BaseModel

class QuestionRequest(BaseModel):
    id: int
    query: str

class AnswerResponse(BaseModel):
    id: int
    query: str
    answer: str

async def dummy_llm_call(query: str, client) -> str:
    response = await client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": query}]
    )
    return response.choices[0].message.content

@pytest.mark.asyncio
async def test_dummy_llm_call_success():
    mock_client = AsyncMock()
    mock_completion = AsyncMock()
    mock_choice = AsyncMock()
    mock_choice.message.content = "Mocked answer payload"
    mock_completion.choices = [mock_choice]
    mock_client.chat.completions.create.return_value = mock_completion

    result = await dummy_llm_call("Test Query", mock_client)
    
    assert result == "Mocked answer payload"
    mock_client.chat.completions.create.assert_called_once()
    print("Test passed successfully!")

# Run unit test directly
await test_dummy_llm_call_success()

### ADR 0002: API Interface Contract Locking

**Context:** Standardizing the interface contract for client applications interacting with the RAG microservice.

**Decision:** All standard interactions will adhere strictly to Pydantic JSON validation schemas.

#### Schema Spec: `/v1/ask`
* **Request (POST):** `{"query": "string (min_length: 3)"}`
* **Response (200 OK):** `{"query": "string", "answer": "string"}`
* **Error Response (400 Bad Request):** `{"detail": "Query string cannot be empty."}`